In [175]:
from langchain_community.document_loaders import TextLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END, START
from typing import Optional, TypedDict

In [176]:
chat_llm = ChatOllama(
    model="llama3.1",
    temperature=0.2
)
embeddings_llm = OllamaEmbeddings(model="nomic-embed-text")

In [177]:
def load_documents(file_path):
    if file_path.endswith(".pdf"):
        loader = PyMuPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)
    return loader.load()

In [178]:
docs = load_documents("../../documents/book.pdf")

In [179]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

In [182]:
vectorstore = FAISS.from_documents(chunks, embeddings_llm)
retriever = vectorstore.as_retriever()

In [ ]:
class ChatState(TypedDict):
    question: str
    intent: Optional[str]

In [ ]:
from enum import Enum


class Intent(str, Enum):
    rag = "rag"
    appointment = "appointment"


class IntentClassify(BaseModel):
    intent: Intent

In [ ]:
class IntentClassify(BaseModel):
    intent: str = Field(description="rag or appointment")

In [ ]:
def classify(graph_state: ChatState):

    prompt = f"""
    You are a strict intent classifier.

    Choose ONLY ONE:
    - rag
    - appointment

    Rules:
    - Output must match exactly one label
    - No explanation
    - No extra text

    User query: {graph_state['question']}
    """

    res = chat_llm.with_structured_output(IntentClassify).invoke(prompt)

    return {"intent": res.intent.value}

In [ ]:
memory = MemorySaver()
graph = StateGraph(ChatState)

In [ ]:

graph.add_node("classify", classify)

# entry
graph.add_edge(START, "classify")
graph.add_edge("classify", END)

In [ ]:
workflow = graph.compile(memory)

In [ ]:
state = {"question": "book an appointment"}
config ={"configurable": {"thread_id": "thread 1"}}


result = workflow.invoke(
    state,
    config=config,
)

In [181]:
result

{'question': 'book an appointment', 'intent': 'make a reservation'}